In [0]:
df_customers = spark.table('ecommerce_dev.bronze.customers')

In [0]:
df_customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_table: string (nullable = true)



In [0]:
display(df_customers.limit(5))

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,_rescued_data,_ingested_at,_source_file,_source_table
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,null,2026-08-04T00:54:52.697Z,/Volumes/ecommerce_dev/raw_data/landing/olist_customers.csv,customers
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,null,2026-08-04T00:54:52.697Z,/Volumes/ecommerce_dev/raw_data/landing/olist_customers.csv,customers
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,null,2026-08-04T00:54:52.697Z,/Volumes/ecommerce_dev/raw_data/landing/olist_customers.csv,customers
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,null,2026-08-04T00:54:52.697Z,/Volumes/ecommerce_dev/raw_data/landing/olist_customers.csv,customers
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,null,2026-08-04T00:54:52.697Z,/Volumes/ecommerce_dev/raw_data/landing/olist_customers.csv,customers


In [0]:
print(f"Row count: {df_customers.count()}")

Row count: 99441


In [0]:
from pyspark.sql import functions as F

# Nulls per column
df_customers.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_customers.columns]
).display()

# Full-row duplicates
print("Full duplicate rows:", df_customers.count() - df_customers.dropDuplicates().count())

# customer_id should be unique — it's the PK
print("Duplicate customer_id:", df_customers.count() - df_customers.dropDuplicates(["customer_id"]).count())

# zip code prefix often comes as int or string with leading zero loss — check
df_customers.select("customer_zip_code_prefix").distinct().orderBy("customer_zip_code_prefix").limit(10).display()

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,_rescued_data,_ingested_at,_source_file,_source_table
0,0,0,0,0,99441,0,0,0


Full duplicate rows: 0
Duplicate customer_id: 0


customer_zip_code_prefix
1003
1004
1005
1006
1007
1008
1009
1011
1012
1013


In [0]:
df_silver_customers = (
    df_customers.withColumn("customer_zip_code_prefix", F.lpad(F.col("customer_zip_code_prefix").cast("string"),5,"0"))
    .withColumn("customer_city", F.initcap(F.trim(F.col("customer_city"))))
    .withColumn("customer_state", F.upper(F.trim(F.col("customer_state"))))
    .drop("_rescued_data", "_source_file", "_source_table")
    .withColumnRenamed("_ingested_at", "bronze_ingested_at")
)

df_silver_customers.limit(5).display()

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,bronze_ingested_at
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,Franca,SP,2026-08-04T00:54:52.697Z
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,Sao Bernardo Do Campo,SP,2026-08-04T00:54:52.697Z
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,Sao Paulo,SP,2026-08-04T00:54:52.697Z
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,Mogi Das Cruzes,SP,2026-08-04T00:54:52.697Z
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,Campinas,SP,2026-08-04T00:54:52.697Z


In [0]:
(
    df_silver_customers.write
        .format('delta')
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable("ecommerce_dev.silver.customers")
)

In [0]:
spark.sql("""
          ALTER TABLE ecommerce_dev.silver.customers
          ALTER COLUMN customer_id SET NOT NULL
          """)

DataFrame[]

In [0]:
spark.sql("""
          ALTER TABLE ecommerce_dev.silver.customers
          ADD CONSTRAINT pk_customer_id PRIMARY KEY (customer_id)
          """)

DataFrame[]

In [0]:
spark.table("ecommerce_dev.silver.customers").select("customer_city").distinct().limit(20).display()

customer_city
Franca
Sao Bernardo Do Campo
Sao Paulo
Mogi Das Cruzes
Campinas
Jaragua Do Sul
Timoteo
Curitiba
Belo Horizonte
Montes Claros


In [0]:
df_silver_customers.limit(5).display()

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,bronze_ingested_at
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,Franca,SP,2026-08-04T00:54:52.697Z
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,Sao Bernardo Do Campo,SP,2026-08-04T00:54:52.697Z
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,Sao Paulo,SP,2026-08-04T00:54:52.697Z
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,Mogi Das Cruzes,SP,2026-08-04T00:54:52.697Z
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,Campinas,SP,2026-08-04T00:54:52.697Z


In [0]:
spark.sql("""
          COMMENT ON TABLE ecommerce_dev.silver.customers IS
          'Cleaned customer dimension source. Zip prefix zero-padded to 5 digits, deduplicated on customer_id, city/state casing standardized. PK: customer_id.'
          """)

DataFrame[]

In [0]:
%sql
DESCRIBE EXTENDED ecommerce_dev.silver.customers;

col_name,data_type,comment
customer_id,string,null
customer_unique_id,string,null
customer_zip_code_prefix,string,null
customer_city,string,null
customer_state,string,null
bronze_ingested_at,timestamp,null
,,
# Delta Statistics Columns,,
Column Names,"customer_id, customer_state, customer_zip_code_prefix, bronze_ingested_at, customer_unique_id, customer_city",
Column Selection Method,first-32,
